# 05. Fitting Comparison

두 개의 저장된 파라미터 JSON을 같은 실측 로그에 다시 적용해, 데이터셋별 지표와 그래프를 한 노트북에서 비교한다.

## 파라미터 기원과 피팅 방식

| label | json | origin notebook | fitting summary |
| --- | --- | --- | --- |
| `full_resolution` | `fitted_motor_params.json` | `04_1_parameter_fitting.ipynb` | Step으로 `delay_steps`를 추정하고, Ramp로 마찰 초기값을 만든 뒤, Multi-sine과 Step/Ramp/Stop 구간을 원본 시간 해상도로 묶어 Newton 전체 응답 loss를 최소화한 결과 |
| `decimated_refinement` | `fitted_motor_params_dec.json` | `04_2_parameter_fitting_dec.ipynb` | 기본 흐름은 동일하지만 refinement 단계에서 Step/Ramp/Multi-sine 로그를 adaptive decimation 해서 샘플 수를 줄이고, 파형 shape를 유지한 상태로 전체 응답 loss를 최소화한 결과 |

## 이 노트북이 하는 일

1. 각 JSON에 저장된 `fit_logs`, `validation_logs`를 읽는다.
2. 같은 실측 CSV에 대해 각 파라미터 세트로 Newton 시뮬레이션을 다시 실행한다.
3. 공통 지표(`RMSE`, `NRMSE`)와 프로파일 전용 지표(step/sine/ramp)를 계산한다.
4. 실측값과 시뮬레이션을 겹쳐 그린 그래프를 데이터셋별로 비교한다.
5. 데이터셋별 승자와 전체 평균 기준의 승자를 요약한다.

참고: 현재 워크스페이스에서는 `fitted_motor_params.json` 안의 `sine_5hz` 파일이 없어 해당 데이터셋은 자동으로 제외된다.


In [ ]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import json
import re
import sys
import warnings

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "code" / "single_motor_twin.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("code/single_motor_twin.py를 찾을 수 없습니다.")

CODE_DIR = PROJECT_ROOT / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from metrics import trajectory_metrics, step_response_metrics, sine_response_metrics, ramp_response_metrics
from single_motor_twin import MotorParams, simulate_motor
from wmx_log_utils import load_wmx_log

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", message="WMX 샘플 간격이 일정하지 않습니다.*")


In [ ]:
PARAMETER_SPECS = [
    {
        "label": "full_resolution",
        "json_path": PROJECT_ROOT / "fitted_motor_params.json",
        "origin_notebook": PROJECT_ROOT / "notebooks" / "04_1_parameter_fitting.ipynb",
        "fitting_method": "원본 해상도 Step/Ramp/Multi-sine/Stop 기반 전체 응답 피팅",
    },
    {
        "label": "decimated_refinement",
        "json_path": PROJECT_ROOT / "fitted_motor_params_dec.json",
        "origin_notebook": PROJECT_ROOT / "notebooks" / "04_2_parameter_fitting_dec.ipynb",
        "fitting_method": "adaptive decimation refinement를 포함한 전체 응답 피팅",
    },
]

DEG_TO_RAD = np.pi / 180.0
RATED_TORQUE_NM = 0.16
LOG_OPTIONS = {
    "cycle_period_s": 1.0e-3,
    "position_scale": DEG_TO_RAD,
    "velocity_scale": DEG_TO_RAD,
    "torque_scale": RATED_TORQUE_NM / 100.0,
}
SIGNAL_KEYS = (
    "command_position",
    "feedback_position",
    "feedback_velocity",
    "feedback_torque",
)
INITIAL_WINDOW_S = 0.02


def load_parameter_bundle(spec):
    payload = json.loads(spec["json_path"].read_text(encoding="utf-8"))
    return {
        **spec,
        "payload": payload,
        "motor_params": MotorParams(**payload["motor_params"]),
    }


def align_log(data, profile):
    raw_time = np.asarray(data["time"], dtype=np.float64)
    dt = float(np.median(np.diff(raw_time)))
    count = int(np.floor((raw_time[-1] - raw_time[0]) / dt + 1.0e-9)) + 1
    time_axis = np.arange(count, dtype=np.float64) * dt
    result = {
        key: np.interp(time_axis, raw_time - raw_time[0], data[key])
        for key in SIGNAL_KEYS
    }
    initial_count = min(count, max(3, int(round(INITIAL_WINDOW_S / dt))))
    position_origin = float(np.median(result["feedback_position"][:initial_count]))
    initial_velocity = float(np.median(result["feedback_velocity"][:initial_count]))
    result["command_position"] -= position_origin
    result["feedback_position"] -= position_origin
    result.update({
        "profile": profile,
        "source": data["source"],
        "time": time_axis,
        "dt": dt,
        "source_dt": dt,
        "position_origin": position_origin,
        "initial_position": float(result["feedback_position"][0]),
        "initial_velocity": initial_velocity,
    })
    return result


def infer_profile_kind(name):
    lowered = name.lower()
    if "step" in lowered:
        return "step"
    if "ramp" in lowered:
        return "ramp"
    if "sine" in lowered:
        return "sine"
    if "scurve" in lowered:
        return "validation"
    return "other"


def infer_frequency_hz(name):
    match = re.search(r"sine_(\d+(?:\.\d+)?)hz", name.lower())
    return float(match.group(1)) if match else None


def build_dataset_table(bundles):
    rows = []
    for bundle in bundles:
        for group_name in ("fit_logs", "validation_logs"):
            for dataset_name, raw_path in bundle["payload"][group_name].items():
                path = Path(raw_path)
                rows.append({
                    "parameter_label": bundle["label"],
                    "group": group_name,
                    "dataset": dataset_name,
                    "path": str(path),
                    "exists": path.exists(),
                })
    return pd.DataFrame(rows)


In [ ]:
bundles = [load_parameter_bundle(spec) for spec in PARAMETER_SPECS]
dataset_status = build_dataset_table(bundles)
dataset_status.sort_values(["dataset", "parameter_label"]).reset_index(drop=True)


In [ ]:
all_dataset_names = []
for bundle in bundles:
    all_dataset_names.extend(bundle["payload"]["fit_logs"].keys())
    all_dataset_names.extend(bundle["payload"]["validation_logs"].keys())
all_dataset_names = list(dict.fromkeys(all_dataset_names))

preferred_paths = {}
for dataset_name in all_dataset_names:
    candidates = []
    for bundle in bundles:
        for group_name in ("fit_logs", "validation_logs"):
            if dataset_name in bundle["payload"][group_name]:
                candidate = Path(bundle["payload"][group_name][dataset_name])
                candidates.append(candidate)
    existing = next((path for path in candidates if path.exists()), None)
    preferred_paths[dataset_name] = existing or candidates[0]

shared_measurements = {}
missing_measurements = []
for dataset_name, dataset_path in preferred_paths.items():
    if not dataset_path.exists():
        missing_measurements.append((dataset_name, str(dataset_path)))
        continue
    measurement = load_wmx_log(dataset_path, **LOG_OPTIONS)
    shared_measurements[dataset_name] = align_log(measurement, dataset_name)

if missing_measurements:
    print("비교에서 제외된 로그")
    for dataset_name, raw_path in missing_measurements:
        print(f"- {dataset_name}: {raw_path}")

available_datasets = sorted(shared_measurements)
available_datasets


In [ ]:
simulations = {}
metric_rows = []
errors = []

for bundle in bundles:
    parameter_label = bundle["label"]
    simulations[parameter_label] = {}
    for dataset_name, measurement in shared_measurements.items():
        try:
            simulation = simulate_motor(
                measurement["command_position"],
                measurement["dt"],
                bundle["motor_params"],
                device="cpu",
                use_viewer=False,
                initial_position=measurement["initial_position"],
                initial_velocity=measurement["initial_velocity"],
            )
        except Exception as exc:
            errors.append({
                "parameter_label": parameter_label,
                "dataset": dataset_name,
                "error": repr(exc),
            })
            continue

        simulations[parameter_label][dataset_name] = simulation
        row = {
            "parameter_label": parameter_label,
            "dataset": dataset_name,
            "kind": infer_profile_kind(dataset_name),
            "frequency_hz": infer_frequency_hz(dataset_name),
            "sample_count": len(measurement["time"]),
            "dt_s": measurement["dt"],
        }
        row.update(trajectory_metrics(measurement, simulation))

        kind = row["kind"]
        if kind == "step":
            row.update(step_response_metrics(measurement, simulation))
        elif kind == "sine" and row["frequency_hz"] is not None:
            row.update(sine_response_metrics(measurement, simulation, row["frequency_hz"]))
        elif kind == "ramp":
            row.update(ramp_response_metrics(measurement, simulation))

        metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows).sort_values(["dataset", "parameter_label"]).reset_index(drop=True)
errors_df = pd.DataFrame(errors)

if not errors_df.empty:
    print("시뮬레이션 실패 항목")
    display(errors_df)

metrics_df


In [ ]:
core_columns = [
    "parameter_label",
    "dataset",
    "kind",
    "position_rmse",
    "position_nrmse",
    "velocity_rmse",
    "velocity_nrmse",
    "torque_rmse",
    "torque_nrmse",
]
summary_df = metrics_df[core_columns].copy()
summary_df = summary_df.sort_values(["dataset", "parameter_label"]).reset_index(drop=True)
summary_df


In [ ]:
ranking_columns = ["position_nrmse", "velocity_nrmse", "torque_nrmse"]
dataset_winners = []
for dataset_name, group in metrics_df.groupby("dataset", sort=True):
    candidate = group.copy()
    candidate["aggregate_nrmse"] = candidate[ranking_columns].mean(axis=1)
    winner = candidate.sort_values("aggregate_nrmse").iloc[0]
    dataset_winners.append({
        "dataset": dataset_name,
        "winner": winner["parameter_label"],
        "aggregate_nrmse": winner["aggregate_nrmse"],
    })
dataset_winners_df = pd.DataFrame(dataset_winners)
dataset_winners_df


In [ ]:
overall_summary = (
    metrics_df.groupby("parameter_label")[[
        "position_nrmse",
        "velocity_nrmse",
        "torque_nrmse",
        "position_rmse",
        "velocity_rmse",
        "torque_rmse",
    ]]
    .mean()
    .assign(aggregate_nrmse=lambda df: df[["position_nrmse", "velocity_nrmse", "torque_nrmse"]].mean(axis=1))
    .sort_values("aggregate_nrmse")
)
overall_summary


In [ ]:
parameter_table = pd.DataFrame([
    {
        "parameter_label": bundle["label"],
        "origin_notebook": bundle["origin_notebook"].name,
        "json_file": bundle["json_path"].name,
        **asdict(bundle["motor_params"]),
    }
    for bundle in bundles
])
parameter_table


## 데이터셋별 그래프

각 데이터셋마다 위쪽은 위치, 가운데는 속도, 아래쪽은 토크를 그린다. 검은색 점선은 command, 굵은 실선은 실측값, 얇은 실선들은 각 JSON 파라미터로 만든 시뮬레이션이다.


In [ ]:
PLOT_SIGNALS = [
    ("feedback_position", "Position [rad]"),
    ("feedback_velocity", "Velocity [rad/s]"),
    ("feedback_torque", "Torque [N m]"),
]

for dataset_name in available_datasets:
    measurement = shared_measurements[dataset_name]
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    fig.suptitle(dataset_name)
    time = measurement["time"]

    for axis, (signal_key, ylabel) in zip(axes, PLOT_SIGNALS):
        if signal_key == "feedback_position":
            axis.plot(time, measurement["command_position"], color="black", linestyle="--", linewidth=1.2, label="command")
        axis.plot(time, measurement[signal_key], linewidth=2.0, label="measured")
        for bundle in bundles:
            parameter_label = bundle["label"]
            simulation = simulations.get(parameter_label, {}).get(dataset_name)
            if simulation is None:
                continue
            axis.plot(time, simulation[signal_key], linewidth=1.2, label=parameter_label)
        axis.set_ylabel(ylabel)
        axis.grid(True, alpha=0.3)
        axis.legend(loc="best")

    axes[-1].set_xlabel("Time [s]")
    plt.tight_layout()
    plt.show()


## 데이터셋별 세부 지표 보기

아래 셀에서 `SELECTED_DATASET` 값을 바꾸면 해당 데이터셋의 모든 지표를 표로 다시 볼 수 있다.


In [ ]:
SELECTED_DATASET = available_datasets[0] if available_datasets else None
if SELECTED_DATASET is None:
    print("사용 가능한 데이터셋이 없습니다.")
else:
    detail_df = metrics_df.loc[metrics_df["dataset"] == SELECTED_DATASET].T
    detail_df.columns = [f"{SELECTED_DATASET}:{label}" for label in metrics_df.loc[metrics_df["dataset"] == SELECTED_DATASET, "parameter_label"]]
    display(detail_df)
